## 📈 Predicting English Rugby Premiership Final Positions Using Probabilistic Modelling & Simulation

**Competition:** English Rugby Premiership 2025/26  
**Purpose:** Estimate probabilities of final league positions using simulation  
**Methods:** Monte Carlo simulation  
**Author:** [Victoria Friss de Kereki](https://www.linkedin.com/in/victoria-friss-de-kereki/)   
**Medium Articles:**   
[Predicting English Rugby Premiership Final Positions Using Probabilistic Modelling & Simulation](https://medium.com/@vickyfrissdekereki/predicting-english-rugby-premiership-final-positions-using-probabilistic-modelling-simulation-353a7810be10)  
[How I Built a Rugby Dataset from Scratch with Web Scraping in Python](https://medium.com/@vickyfrissdekereki/how-i-built-a-rugby-dataset-from-scratch-with-web-scraping-in-python-9de4c6de95b2)  
[Predicting English Rugby Premiership Final Positions: A Technical Walkthrough in Python](https://medium.com/@vickyfrissdekereki/predicting-english-rugby-premiership-final-positions-a-technical-walkthrough-in-python-1d4a7a955375)

---

**Notebook first written:** `30/01/2026`  
**Last updated:** `23/04/2026`  

> This notebook develops a probabilistic framework to predict final English Rugby Premiership final positions using past matches breakdown to build probabilities of each outcome for each remaining match, which are then used to simulate the remainder of the season via Monte Carlo methods, generating distributions over final points totals and league positions.
>
> The analysis focuses on estimating the likelihood of key outcomes such as title wins, top-four finishes (qualification to play-offs), etc. Results are presented at team level with uncertainty intervals, and the framework can be extended to incorporate form, fixture difficulty, or alternative predictive inputs beyond betting markets.


<div style="text-align: left;">
    <img src="Images and others/Predicting English Rugby Premiership Final Positions Using Probabilistic Modelling & Simulation.png"  
         alt="Predicting English Rugby Premiership Final Positions Using Probabilistic Modelling & Simulation"  
         width="600">
</div>

In [1]:
# Core
from datetime import datetime, timedelta
import os

# Data manipulation
import numpy as np
import pandas as pd

# APIs & environment
import requests
from dotenv import load_dotenv

# Statistics
from scipy.stats import poisson

# Visualisation
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

## 1. Premiership Final Standings (ESPN Scraping)
##### Adapting the ESPN scraper I built in my previous project to be used for Rugby Premiership instead of Football Premier League.

In [2]:
url = "https://www.espn.com/rugby/standings/_/league/267979"
tables = pd.read_html(url)

# Fix ESPN header issue
teams_raw = tables[0].copy()

# Move column names into first row
teams_raw.loc[-1] = teams_raw.columns
teams_raw.index = teams_raw.index + 1
teams_raw = teams_raw.sort_index().reset_index(drop=True)

# Stats table is fine as-is
stats = tables[1]

# Parse team table
teams = pd.DataFrame()

teams["position"] = (
    teams_raw.iloc[:, 0]
    .astype(str)
    .str.extract(r"^(\d+)")
    .astype(int)
)

teams["team"] = (
    teams_raw.iloc[:, 0]
    .astype(str)
    .str.replace(r"^\d+", "", regex=True)        # remove position
    .str.replace(r"^[A-Z]{2,3}", "", regex=True) # remove abbreviation
    .str.strip()
)

# Parse stats table
stats.columns = [
    "gp", "w", "d", "l", "bye",
    "pf", "pa", "tf", "ta",
    "tbp", "lbp", "bp",
    "pd", "pts"
]

stats = stats.apply(
    lambda c: (
        c.astype(str)
         .str.replace("+", "", regex=False)
         .astype(int)
    )
)

# Combine
premiership = pd.concat([teams, stats], axis=1)

premiership.to_csv('premiership.csv', index=False)

premiership

,position,team,gp,w,d,l,bye,pf,pa,tf,ta,tbp,lbp,bp,pd,pts
0,1,Northampton Saints,13,11,1,1,0,474,354,70,49,11,0,11,120,57
1,2,Bath Rugby,13,11,0,2,0,484,293,72,42,11,1,12,191,56
2,3,Leicester Tigers,13,10,0,3,0,432,277,61,37,10,1,11,155,51
3,4,Exeter Chiefs,13,8,1,4,0,364,242,51,33,9,4,13,122,47
4,5,Bristol Rugby,13,9,0,4,0,376,298,53,42,6,1,7,78,43
5,6,Saracens,13,6,0,7,0,500,350,73,50,10,4,14,150,38
6,7,Sale Sharks,13,3,0,10,0,344,439,48,65,6,4,10,-95,22
7,8,Gloucester Rugby,13,2,0,11,0,269,443,41,61,5,3,8,-174,16
8,9,Harlequins,13,3,0,10,0,248,439,33,66,2,1,3,-191,15
9,10,Newcastle Falcons,13,1,0,12,0,211,567,29,86,2,1,3,-356,7


## 2. Get betting odds using API

#### No matches betting odds available as there is still over a month to go until the next match.

## 3. Get fixtures for upcoming EPL games
#### No API available for Premiership Rugby, so performing web scraping from https://www.skysports.com/rugby-union/competitions/gallagher-prem/fixtures instead.

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.skysports.com/rugby-union/competitions/gallagher-prem/fixtures"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

fixtures_container = soup.find("div", class_="fixres__body")

month = None
date = None

dates = []
months = []
home_teams = []
away_teams = []
times = []

for elem in fixtures_container.children:
    # Skip empty elements or strings
    if not hasattr(elem, "name"):
        continue

    # Month header
    if elem.name == "h3" and "fixres__header1" in elem.get("class", []):
        month = elem.get_text(strip=True)

    # Date header
    elif elem.name == "h4" and "fixres__header2" in elem.get("class", []):
        date = elem.get_text(strip=True)

    # Match item
    elif elem.name == "div" and "fixres__item" in elem.get("class", []):
        home = elem.select_one("span.matches__participant--side1 span.swap-text__target").get_text(strip=True)
        away = elem.select_one("span.matches__participant--side2 span.swap-text__target").get_text(strip=True)
        time = elem.select_one("span.matches__date").get_text(strip=True)

        months.append(month)
        dates.append(date)
        home_teams.append(home)
        away_teams.append(away)
        times.append(time)

# Build DataFrame
df_future_1 = pd.DataFrame({
    "month": months,
    "date": dates,
    "time": times,
    "home_team": home_teams,
    "away_team": away_teams
})

print(df_future_1.head())
print("Total fixtures scraped:", len(df_future_1))

        month                 date   time            home_team  \
0  April 2026    Friday 24th April  19:45  Newcastle Red Bulls   
1  April 2026  Saturday 25th April  15:00           Harlequins   
2  April 2026  Saturday 25th April  15:05             Saracens   
3  April 2026  Saturday 25th April  17:30          Northampton   
4  April 2026    Sunday 26th April  15:30           Gloucester   

          away_team  
0     Bristol Bears  
1       Sale Sharks  
2  Leicester Tigers  
3              Bath  
4            Exeter  
Total fixtures scraped: 28


In [4]:
# Remove fixtures with TBC in either team
df_future_1 = df_future_1[~df_future_1["home_team"].str.contains("TBC") & ~df_future_1["away_team"].str.contains("TBC")].reset_index(drop=True)
df_future = df_future_1.copy()
print("Total fixtures after removing TBC matches:", len(df_future))

Total fixtures after removing TBC matches: 25


In [5]:
df_future.to_csv('df_future.csv', index=False)
df_future

,month,date,time,home_team,away_team
0,April 2026,Friday 24th April,19:45,Newcastle Red Bulls,Bristol Bears
1,April 2026,Saturday 25th April,15:00,Harlequins,Sale Sharks
2,April 2026,Saturday 25th April,15:05,Saracens,Leicester Tigers
3,April 2026,Saturday 25th April,17:30,Northampton,Bath
4,April 2026,Sunday 26th April,15:30,Gloucester,Exeter
5,May 2026,Friday 8th May,19:45,Gloucester,Sale Sharks
6,May 2026,Saturday 9th May,15:05,Leicester Tigers,Northampton
7,May 2026,Saturday 9th May,17:30,Bristol Bears,Saracens
8,May 2026,Sunday 10th May,15:00,Newcastle Red Bulls,Harlequins
9,May 2026,Sunday 10th May,15:30,Exeter,Bath


## 4. Get this season (2025/26) and last 2 seasons (2024/25 & 2023/24) results
### Again, no API available here. Will do web scraping from https://www.tntsports.co.uk/rugby/premiership/calendar-results.shtml to get tries and full breakdown.

In [6]:
import re

def scrape_match_stats(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to load {url}")
        return None
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Extract season from URL
    match = re.search(r'/premiership/(\d{4}-\d{4})/', url)
    season = match.group(1) if match else None
    
    # Team names
    team_tags = soup.select('a[data-testid="atom-participant-banner"] div.overflow-hidden')
    if len(team_tags) != 2:
        print(f"Couldn't find team names for {url}")
        return None
    
    home_team = team_tags[0].text.strip()
    away_team = team_tags[1].text.strip()
    
    # Team scores
    score_tags = soup.select('div[data-testid="team-match-header-score-atom-container"] div[data-testid="match-score-atom-score-content-box-score"]')
    if len(score_tags) == 2:
        home_score = int(score_tags[0].text.strip())
        away_score = int(score_tags[1].text.strip())
    else:
        home_score = away_score = None
    
    # Match date
    date_tag = soup.select_one('div[data-testid="molecule-match-header-top"] div.caption-2.text-neutral-05')
    if date_tag:
        date_text = date_tag.text.strip()
        match_date = date_text.split("/")[-1].strip() if "/" in date_text else None
    else:
        match_date = None
    
    # Stats rows
    stats_rows = soup.select('div[data-testid="molecule-team-action-row"]')
    stats_dict = {}
    
    for row in stats_rows:
        stat_name_tag = row.find('span')
        if not stat_name_tag:
            continue
        stat_name = stat_name_tag.text.strip()
        
        participants = row.select('div[data-testid="molecule-team-action-row-participant"]')
        if len(participants) != 2:
            continue
        
        home_val = int(participants[0].find('div', class_='caps-s6-fx').text.strip())
        away_val = int(participants[1].find('div', class_='caps-s6-fx').text.strip())
        
        stats_dict[stat_name] = [home_val, away_val]
    
    return {
        "season": season,
        "date": match_date,
        "home_team": home_team,
        "away_team": away_team,
        "home_score": home_score,
        "away_score": away_score,
        **stats_dict
    } 

##### Generate list of URLs

In [7]:
season_match_ranges = {
    "2023-2024": {"start": 1457678, "end": 1457767},  # 90
    "2024-2025": {"start": 1548080, "end": 1548169},  # 90
    "2025-2026": {"start": 1628800, "end": 1628844},  # 45 - first half of current season
}


urls = []
season_counters = {s: 0 for s in season_match_ranges}

for _, row in all_results.iterrows():
    season = row["season"]
    start_year, end_year = season.split("-")
    season_str = f"{start_year}-20{end_year}"

    if season_str not in season_match_ranges:
        continue

    start = season_match_ranges[season_str]["start"]
    end = season_match_ranges[season_str]["end"]

    match_number = start + season_counters[season_str]

    # stop once season is full
    if match_number > end:
        continue

    url = (
        f"https://www.tntsports.co.uk/rugby/premiership/"
        f"{season_str}/live-match_mtc{match_number}/live.shtml"
    )

    urls.append(url)
    season_counters[season_str] += 1

NameError: name 'all_results' is not defined

In [ ]:
import time

all_matches_stats = []

total = len(urls)

for i, url in enumerate(urls, start=1):
    print(f"[{i}/{total}] Scraping: {url}")

    try:
        match = scrape_match_stats(url)

        if match:
            all_matches_stats.append(match)
            print(f"    ✅ Success: {match['home_team']} vs {match['away_team']}")
        else:
            print("    ⚠️  No data returned")

    except Exception as e:
        print(f"    ❌ Error scraping {url}: {e}")

    # polite delay to avoid hammering the site
    time.sleep(1)

print("\nFinished scraping.")
print(f"Total successful matches: {len(all_matches_stats)}")

In [ ]:
# Convert list of dicts to DataFrame
df_matches = pd.DataFrame(all_matches_stats)

# Split stats lists into separate home/away columns
for stat in ['Tries', 'Conversions', 'Penalties', 'Drops']:
    df_matches[[f'{stat}_home', f'{stat}_away']] = pd.DataFrame(df_matches[stat].tolist(), index=df_matches.index)
    df_matches.drop(columns=[stat], inplace=True)

# Reorder columns
df_matches = df_matches[
    ['season', 'date', 'home_team', 'away_team', 
     'home_score', 'away_score',
     'Tries_home', 'Tries_away',
     'Conversions_home', 'Conversions_away',
     'Penalties_home', 'Penalties_away',
     'Drops_home', 'Drops_away']
]

# Save to CSV
df_matches.to_csv('rugby_matches_stats.csv', index=False)

df_matches['season'].value_counts().sort_index().head()

##### Some basic quality checks

In [ ]:
df_matches['season'].value_counts().sort_index()

In [ ]:
# Count home matches
home_counts = df_matches['home_team'].value_counts()

# Count away matches
away_counts = df_matches['away_team'].value_counts()

# Combine them
total_counts = home_counts.add(away_counts, fill_value=0).astype(int).sort_values(ascending=False)

print(total_counts)

## 5. Combine and calculate probabilities of TRIES, CONVERSIONS, PENALTIES, TRY BONUS POINT, & LOSING BONUS POINT

In [ ]:
past_premierships = df_matches.copy()

In [ ]:
# Add weights: more recent games = more weight
past_premierships["weight"] = np.linspace(1, 2, len(past_premierships))  # simple linear weighting

In [ ]:
past_premierships.tail()

In [ ]:
home_advantage_tries = past_premierships['Tries_home'].mean() - past_premierships['Tries_away'].mean()
print("Home advantage (average tries):", home_advantage_tries)

home_advantage_penalties = past_premierships['Penalties_home'].mean() - past_premierships['Penalties_away'].mean()
print("Home advantage (average penalties):", home_advantage_penalties)      
# NO HOME ADVANTAGE FOR PENALTIES. Value is small and even negative, will be ignored.

In [ ]:
import pandas as pd

# All teams
teams = pd.unique(past_premierships[["home_team", "away_team"]].values.ravel("K"))

# Initialize
team_stats = {}
attack_points = pd.Series(1.0, index=teams)
defense_points = pd.Series(1.0, index=teams)
attack_tries = pd.Series(1.0, index=teams)
defense_tries = pd.Series(1.0, index=teams)

for team in teams:
    home_games = past_premierships[past_premierships["home_team"] == team]
    away_games = past_premierships[past_premierships["away_team"] == team]

    # Points scored/conceded
    points_scored = (home_games["home_score"]*home_games["weight"]).sum() + \
                    (away_games["away_score"]*away_games["weight"]).sum()
    points_conceded = (home_games["away_score"]*home_games["weight"]).sum() + \
                      (away_games["home_score"]*away_games["weight"]).sum()

    # Tries scored/conceded
    tries_scored = (home_games["Tries_home"]*home_games["weight"]).sum() + \
                   (away_games["Tries_away"]*away_games["weight"]).sum()
    tries_conceded = (home_games["Tries_away"]*home_games["weight"]).sum() + \
                     (away_games["Tries_home"]*away_games["weight"]).sum()

    # Weighted matches
    matches = home_games["weight"].sum() + away_games["weight"].sum()

    # Save per game stats
    team_stats[team] = {
        "points_scored_per_game": points_scored/matches,
        "points_conceded_per_game": points_conceded/matches,
        "tries_scored_per_game": tries_scored/matches,
        "tries_conceded_per_game": tries_conceded/matches
    }

# Total points and tries
#total_points = past_premierships["home_score"].sum() + past_premierships["away_score"].sum()
total_tries  = past_premierships["Tries_home"].sum() + past_premierships["Tries_away"].sum()
# Total games (each row = 1 game)
total_games = len(past_premierships)
# League average per team per game
#league_avg_points = total_points / (2 * total_games)
league_avg_tries  = total_tries / (2 * total_games)

# Relative scores
for team in teams:
#    attack_points[team] = team_stats[team]["points_scored_per_game"]
#    defense_points[team] = team_stats[team]["points_conceded_per_game"]
    attack_tries[team] = team_stats[team]["tries_scored_per_game"]/league_avg_tries
    defense_tries[team] = team_stats[team]["tries_conceded_per_game"]/league_avg_tries

# Weighted conversion rate per team
teams_conv = pd.concat([
    past_premierships[['home_team','Tries_home','Conversions_home','weight']].rename(
        columns={'home_team':'team','Tries_home':'tries','Conversions_home':'conversions'}
    ),
    past_premierships[['away_team','Tries_away','Conversions_away','weight']].rename(
        columns={'away_team':'team','Tries_away':'tries','Conversions_away':'conversions'}
    )
])
team_conversion = teams_conv.groupby('team').apply(
    lambda x: (x['conversions']*x['weight']).sum() / (x['tries']*x['weight']).sum()
)

# League average per team per game - penalties and drops
total_penalties = past_premierships["Penalties_home"].sum() + past_premierships["Penalties_away"].sum()
league_avg_penalties  = total_penalties / (2 * total_games) 
total_drops = past_premierships["Drops_home"].sum() + past_premierships["Drops_away"].sum()
league_avg_drops  = total_drops / (2 * total_games)


# --- Relative penalties and drops (offensive) ---
penalties_per_game_rel = team_penalty_drop['penalties_per_game'] / league_avg_penalties
drops_per_game_rel     = team_penalty_drop['drops_per_game'] / league_avg_drops

# --- Relative penalties and drops (defensive) ---
penalties_conceded_per_game_rel = team_penalty_drop_def['penalties_conceded_per_game'] / league_avg_penalties
drops_conceded_per_game_rel     = team_penalty_drop_def['drops_conceded_per_game'] / league_avg_drops

# --- Combine all into team_strengths ---
team_strengths = pd.DataFrame({
    "attack_tries": attack_tries,
    "defense_tries": defense_tries,
    "conversion_rate": team_conversion,
    "attack_penalties": penalties_per_game_rel,
    "attack_drops": drops_per_game_rel,
    "defense_penalties": penalties_conceded_per_game_rel,
    "defense_drops": drops_conceded_per_game_rel
})

team_strengths = team_strengths.sort_values("attack_tries", ascending=False)
team_strengths

## 6. Model each component with Poisson

In [ ]:
# --- Team name mapping ---
team_name_map = {
    "Bath": "Bath Rugby",
    "Bath Rugby": "Bath Rugby",
    "Exeter": "Exeter Chiefs",
    "Exeter Chiefs": "Exeter Chiefs",
    "Northampton": "Northampton Saints",
    "Northampton Saints": "Northampton Saints",
    "Newcastle Red Bulls": "Newcastle Red Bulls",
    "Newcastle Falcons": "Newcastle Red Bulls", # for premiership table
    "Bristol Bears": "Bristol Bears",
    "Bristol Rugby": "Bristol Bears",
    "Saracens": "Saracens",
    "Leicester Tigers": "Leicester Tigers",
    "Sale Sharks": "Sale Sharks",
    "Gloucester": "Gloucester Rugby",
    "Gloucester Rugby": "Gloucester Rugby",
    "Harlequins": "Harlequins"
}

# --- Apply mapping to df_future ---
df_future['home_team'] = df_future['home_team'].map(team_name_map)
df_future['away_team'] = df_future['away_team'].map(team_name_map)

# --- Apply mapping to premiership table
premiership['team'] = premiership['team'].map(team_name_map)

In [ ]:
import pandas as pd
from scipy.stats import poisson

# --- FULL JOINT OUTCOMES SIMULATION ---
def simulate_match_joint(home_team, away_team, team_strengths, sims=10000):
    """
    Simulate a match multiple times and return joint probabilities for
    all 20 mutually exclusive outcomes:
    - 16 for win combinations (winner's bonus × loser's bonus)
    - 4 for draws (draw with/without bonus combinations)
    """
    # Get team stats
    home = team_strengths.loc[home_team]
    away = team_strengths.loc[away_team]

    home_tries_exp = home['attack_tries'] * away['defense_tries'] * home_advantage_tries * league_avg_tries
    away_tries_exp = away['attack_tries'] * home['defense_tries'] * league_avg_tries

    home_pen_exp = home['attack_penalties'] * away['defense_penalties'] * league_avg_penalties
    away_pen_exp = away['attack_penalties'] * home['defense_penalties'] * league_avg_penalties

    home_drop_exp = home['attack_drops'] * away['defense_drops'] * league_avg_drops
    away_drop_exp = away['attack_drops'] * home['defense_drops'] * league_avg_drops
    
    exp = {
        'home_tries': home_tries_exp,
        'away_tries': away_tries_exp,
        'home_converted': home_tries_exp * home['conversion_rate'],
        'away_converted': away_tries_exp * away['conversion_rate'],
        'home_pen': home_pen_exp,
        'away_pen': away_pen_exp,
        'home_drop': home_drop_exp,
        'away_drop': away_drop_exp
    }

    # Initialize all 20 outcomes
    outcome_counts = {f'{h}_{a}': 0 for h in
                      ['win_no_bonus','win_bonus','lose_no_bonus','lose_try_bonus','lose_close_bonus','lose_bonus2','draw_no_bonus','draw_bonus']
                      for a in
                      ['win_no_bonus','win_bonus','lose_no_bonus','lose_try_bonus','lose_close_bonus','lose_bonus2','draw_no_bonus','draw_bonus']}
    
    # Keep only the 20 valid combinations
    valid_keys = [
        'win_no_bonus_lose_no_bonus','win_no_bonus_lose_try_bonus','win_no_bonus_lose_close_bonus','win_no_bonus_lose_bonus2',
        'win_bonus_lose_no_bonus','win_bonus_lose_try_bonus','win_bonus_lose_close_bonus','win_bonus_lose_bonus2',
        'lose_no_bonus_win_no_bonus','lose_no_bonus_win_bonus','lose_try_bonus_win_no_bonus','lose_try_bonus_win_bonus',
        'lose_close_bonus_win_no_bonus','lose_close_bonus_win_bonus','lose_bonus2_win_no_bonus','lose_bonus2_win_bonus',
        'draw_no_bonus_draw_no_bonus','draw_no_bonus_draw_bonus','draw_bonus_draw_no_bonus','draw_bonus_draw_bonus'
    ]
    outcome_counts = {k: 0 for k in valid_keys}

    # --- Run simulations ---
    for _ in range(sims):
        home_tries = poisson.rvs(exp['home_tries'])
        away_tries = poisson.rvs(exp['away_tries'])
        home_converted = min(home_tries, poisson.rvs(exp['home_converted']))
        away_converted = min(away_tries, poisson.rvs(exp['away_converted']))
        home_pen = poisson.rvs(exp['home_pen'])
        away_pen = poisson.rvs(exp['away_pen'])
        home_drop = poisson.rvs(exp['home_drop'])
        away_drop = poisson.rvs(exp['away_drop'])

        home_points = home_tries*5 + home_converted*2 + home_pen*3 + home_drop*3
        away_points = away_tries*5 + away_converted*2 + away_pen*3 + away_drop*3

        home_bonus_try = home_tries >= 4
        away_bonus_try = away_tries >= 4
        home_bonus_close = (away_points - home_points) <= 7 and home_points < away_points
        away_bonus_close = (home_points - away_points) <= 7 and away_points < home_points

        # Determine outcomes
        if home_points > away_points:
            home_outcome = 'win_bonus' if home_bonus_try else 'win_no_bonus'
            if away_bonus_try and away_bonus_close:
                away_outcome = 'lose_bonus2'
            elif away_bonus_try:
                away_outcome = 'lose_try_bonus'
            elif away_bonus_close:
                away_outcome = 'lose_close_bonus'
            else:
                away_outcome = 'lose_no_bonus'
        elif home_points < away_points:
            away_outcome = 'win_bonus' if away_bonus_try else 'win_no_bonus'
            if home_bonus_try and home_bonus_close:
                home_outcome = 'lose_bonus2'
            elif home_bonus_try:
                home_outcome = 'lose_try_bonus'
            elif home_bonus_close:
                home_outcome = 'lose_close_bonus'
            else:
                home_outcome = 'lose_no_bonus'
        else:  # draw
            home_outcome = 'draw_bonus' if home_bonus_try else 'draw_no_bonus'
            away_outcome = 'draw_bonus' if away_bonus_try else 'draw_no_bonus'

        outcome_counts[f'{home_outcome}_{away_outcome}'] += 1

    # Convert counts to probabilities
    return {k: v / sims for k, v in outcome_counts.items()}

In [ ]:
simulate_match_joint("Bath Rugby", "Bristol Bears", team_strengths, sims=10000)

In [ ]:
# List of all 20 joint outcome keys
joint_keys = [
    'win_no_bonus_lose_no_bonus','win_no_bonus_lose_try_bonus','win_no_bonus_lose_close_bonus','win_no_bonus_lose_bonus2',
    'win_bonus_lose_no_bonus','win_bonus_lose_try_bonus','win_bonus_lose_close_bonus','win_bonus_lose_bonus2',
    'lose_no_bonus_win_no_bonus','lose_no_bonus_win_bonus','lose_try_bonus_win_no_bonus','lose_try_bonus_win_bonus',
    'lose_close_bonus_win_no_bonus','lose_close_bonus_win_bonus','lose_bonus2_win_no_bonus','lose_bonus2_win_bonus',
    'draw_no_bonus_draw_no_bonus','draw_no_bonus_draw_bonus','draw_bonus_draw_no_bonus','draw_bonus_draw_bonus'
]

def predict_fixtures_joint(df_future, team_strengths, sims=5000):
    """
    Run simulate_match_joint for each future fixture, 
    returning a DataFrame where each of the 20 outcomes is a column.
    """
    results = []

    for _, row in df_future.iterrows():
        home = row['home_team']
        away = row['away_team']

        # Run joint simulation
        outcome_probs = simulate_match_joint(home, away, team_strengths, sims)

        # Make sure all keys are present
        row_data = row.to_dict()
        for k in joint_keys:
            row_data[k] = outcome_probs.get(k, 0.0)

        results.append(row_data)

    return pd.DataFrame(results)

In [ ]:
np.random.seed(14)
df_predictions = predict_fixtures_joint(df_future, team_strengths, sims=10000)

In [ ]:
df_predictions.head()

## 7. Run simulations to build the Premiership table probabilities

In [ ]:
def simulate_once_rugby(fixtures, table):
    table_sim = table.copy()
    points = dict(zip(table_sim["team"], table_sim["pts"]))

    # List of all 20 outcome columns in your fixtures DataFrame
    outcome_keys = [
    'win_no_bonus_lose_no_bonus','win_no_bonus_lose_try_bonus','win_no_bonus_lose_close_bonus','win_no_bonus_lose_bonus2',
    'win_bonus_lose_no_bonus','win_bonus_lose_try_bonus','win_bonus_lose_close_bonus','win_bonus_lose_bonus2',
    'lose_no_bonus_win_no_bonus','lose_no_bonus_win_bonus','lose_try_bonus_win_no_bonus','lose_try_bonus_win_bonus',
    'lose_close_bonus_win_no_bonus','lose_close_bonus_win_bonus','lose_bonus2_win_no_bonus','lose_bonus2_win_bonus',
    'draw_no_bonus_draw_no_bonus','draw_no_bonus_draw_bonus','draw_bonus_draw_no_bonus','draw_bonus_draw_bonus'
    ]


    # Mapping outcome columns to points (home points, away points)
    points_map = {
        # Home wins
        'win_no_bonus_lose_no_bonus': (4, 0),
        'win_no_bonus_lose_try_bonus': (4, 1),
        'win_no_bonus_lose_close_bonus': (4, 1),
        'win_no_bonus_lose_bonus2': (4, 2),
        'win_bonus_lose_no_bonus': (5, 0),
        'win_bonus_lose_try_bonus': (5, 1),
        'win_bonus_lose_close_bonus': (5, 1),
        'win_bonus_lose_bonus2': (5, 2),
    
        # Home loses
        'lose_no_bonus_win_no_bonus': (0, 4),
        'lose_no_bonus_win_bonus': (0, 5),
        'lose_try_bonus_win_no_bonus': (1, 4),
        'lose_try_bonus_win_bonus': (1, 5),
        'lose_close_bonus_win_no_bonus': (1, 4),
        'lose_close_bonus_win_bonus': (1, 5),
        'lose_bonus2_win_no_bonus': (2, 4),
        'lose_bonus2_win_bonus': (2, 5),
    
        # Draws
        'draw_no_bonus_draw_no_bonus': (2, 2),
        'draw_no_bonus_draw_bonus': (2, 3),
        'draw_bonus_draw_no_bonus': (3, 2),
        'draw_bonus_draw_bonus': (3, 3)
    }

    for _, row in fixtures.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        probs = [row[k] for k in outcome_keys]
        outcome = np.random.choice(outcome_keys, p=probs)

        home_pts, away_pts = points_map[outcome]
        points[home] += home_pts
        points[away] += away_pts

    result_df = table_sim.copy()
    result_df["pts"] = result_df["team"].map(points)
    result_df = result_df.sort_values(["pts"], ascending=False)
    result_df["position"] = np.arange(1, len(result_df)+1)

    return result_df


# -------------------------------
# RUN MULTIPLE SIMULATIONS
# -------------------------------
def run_simulations_rugby(fixtures, table, n_sim=10000):
    """
    Run many simulations to get probability distribution of finishing positions.
    Returns a DataFrame: rows=positions, columns=teams, values=counts.
    """
    position_counts = {team: np.zeros(len(table)) for team in table["team"]}

    for _ in range(n_sim):
        final_table = simulate_once_rugby(fixtures, table)
        for _, row in final_table.iterrows():
            position_counts[row["team"]][row["position"]-1] += 1

    pos_df = pd.DataFrame(position_counts, index=np.arange(1, len(table)+1))
    pos_df.index.name = "position"
    return pos_df


In [ ]:
np.random.seed(14)

n_sim=10000

# Run simulations
position_counts = run_simulations_rugby(df_predictions, premiership, n_sim=n_sim)

# Convert counts to probabilities
position_probs = position_counts / n_sim

In [ ]:
position_probs.index.name = "TEAM"
position_distribution_t = position_probs.T

In [ ]:
position_distribution_pct = position_distribution_t.div(
    position_distribution_t.sum(axis=1),
    axis=0
) * 100


In [ ]:
position_distribution_pct

## 8. Preview and present the results graphically

In [ ]:
# Build label mapping: "position  team" (extra space for 1-9)
team_labels = (
    premiership[["team", "position"]]
    .set_index("team")["position"]
    .map(lambda pos: f"{pos}{'  ' if pos < 10 else ' '}")
)

# Join position and team name into one label
team_labels = (
    premiership[["team", "position"]]
    .assign(
        label=lambda df: df.apply(
            lambda r: f"{r['position']}{'&nbsp;&nbsp;&nbsp;&nbsp;' if r['position'] < 10 else '&nbsp;&nbsp;'}{r['team']}",
            axis=1
        )
    )
    .set_index("team")["label"]
)


# Apply labels to your table index
position_distribution_pct.index = position_distribution_pct.index.map(team_labels)

# Drop position column if present
position_distribution_pct = position_distribution_pct.drop(columns=["position"], errors="ignore")

# Remove index name
position_distribution_pct.index.name = None

In [ ]:
# Copy to avoid modifying original
position_distribution_pct = position_distribution_pct.copy()

# Prepend points from premiership table to team names
position_distribution_pct.index = [
    f"{team} ({pts})" 
    for team, pts in zip(position_distribution_pct.index, premiership["pts"])
]

In [ ]:
greens = plt.cm.Greens
green_cmap = LinearSegmentedColormap.from_list(
    "Greens_soft",
    greens(np.linspace(0.03, 0.65, 256))
)

vmax = 45

def zero_style(val):
    if val < 0.005:
        return "background-color: white !important;"
    return ""

# ---- transform ONLY for colouring ----
color_data = position_distribution_pct.copy()
color_data = (color_data / vmax).pow(0.65) * vmax

position_distribution_pct.style \
    .background_gradient(
        cmap=green_cmap,
        vmin=0,
        vmax=vmax,
        gmap=color_data,
        axis=None          
    ) \
    .applymap(zero_style) \
    .format("{:.2f}%") \
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#e6edf4"),
            ("color", "#333"),
            ("text-align", "center"),
            ("font-family", "Inter, Roboto, Arial, sans-serif"),
            ("font-size", "13px"),
            ("font-weight", "600")
        ]},

        {"selector": "th.col_heading", "props": [
            ("text-align", "center")
        ]},

        {"selector": "th.row_heading", "props": [
            ("text-align", "left"),
            ("font-size", "13px"),
            ("font-weight", "600"),
            ("white-space", "nowrap"),
            ("max-width", "250px"),
            ("overflow", "hidden"),
            ("text-overflow", "ellipsis")
        ]},

        {"selector": "tr:nth-child(odd) th.row_heading", "props": [
            ("background-color", "#fbfcfe")
        ]},
        {"selector": "tr:nth-child(even) th.row_heading", "props": [
            ("background-color", "#e6edf4")
        ]},

        {"selector": "td", "props": [
            ("text-align", "center"),
            ("font-family", "Inter, Roboto, Arial, sans-serif"),
            ("font-size", "12px"),
            ("font-weight", "500"),
            ("color", "#000")
        ]}
    ])
